In [14]:
import torch

In [16]:
import torch.nn as nn

basic start of how to code kv cache
- we need to define a class
- it is like “I am defining a custom PyTorch attention layer/module.”


- __init__ is something like
- “Setup everything needed for this model.”
- it will directly get assigned when the function is called


dim “Each token embedding has 64 values.”
and 4 heads means it sees each token in 4 diff ways
- it captures 4 diff meanings


- Full token embedding:

64 numbers

- Split into 4 heads:

Head1 → 16 dims

Head2 → 16 dims

Head3 → 16 dims

Head4 → 16 dims

In [25]:
class implementingKvCache(nn.Module):
  def __init__(self, dim=64, num_heads=4):
    super.__init__()
    self.dim = dim
    self.num_heads = num_heads
    self.head_dim = dim//num_heads
    self.q_proj = nn.Linear(dim, dim)
    self.q_proj = nn.Linear(dim, dim)
    self.k_proj = nn.Linear(dim, dim)
    self.v_proj = nn.Linear(dim, dim)
    self.out_proj = nn.Linear(dim, dim)
            # THE KV CACHE — starts empty
        # We will fill this as we generate tokens
    self.k_cache = None  # will store all past Keys
    self.v_cache = None  # will store all past Values






    Because dimensions must be whole numbers.

Example:

64 // 4 = 16

not:

16.0
“Take the full embedding vector and divide it evenly among the attention heads.”


Query = “what am I searching for?”

Key   = “what information do I contain?”

Value = “what information should I pass?”

Suppose:

x.shape = (2, 5, 64)

This means:

batch = 2
seq_len = 5
dim = 64

Because code doesn't need that variable later.

_ means:

“Ignore this value.”

k = torch.cat([self.k_cache, k], dim=1)

means:

“Append new key vector to old cached keys.”

So appending along dim=1 means:
Add more tokens to sequence

NOT:

new batches

new features

In [26]:
def forward(self,x):
          # x shape: (batch, seq_len, dim)
  batch, seq_len, _ = x.shape

        # Compute Q, K, V for current token(s)
  q = self.q_proj(x)
  k = self.k_proj(x)
  v = self.v_proj(x)
  if self.k_cache is not None:
            # Append new keys and values to cache
            # dim=1 means we are appending along the sequence dimension
    k = torch.cat([self.k_cache, k], dim=1)
    v = torch.cat([self.v_cache, v], dim=1)
  self.k_cache = k
  self.v_cache = v



In [29]:
def forward(self, x):
        # x shape: (batch, seq_len, dim)
  batch, seq_len, _ = x.shape

        # Compute Q, K, V for current token(s)
  q = self.q_proj(x)
  k = self.k_proj(x)
  v = self.v_proj(x)

        # ============================================================
        # THIS IS THE KEY PART — KV CACHE LOGIC
        # If cache exists, APPEND new K and V to existing cache
        # Instead of recomputing everything from scratch
        # ============================================================

  if self.k_cache is not None:
            # Append new keys and values to cache
            # dim=1 means we are appending along the sequence dimension
    k = torch.cat([self.k_cache, k], dim=1)
    v = torch.cat([self.v_cache, v], dim=1)

        # Update cache with current K and V
  self.k_cache = k
  self.v_cache = v

        # Now compute attention using ALL keys and values
        # (current + all previous from cache)
  scale = self.head_dim ** 0.5
  attn_weights = torch.bmm(q, k.transpose(1, 2)) / scale
  attn_weights = torch.softmax(attn_weights, dim=-1)
  output = torch.bmm(attn_weights, v)

  return self.out_proj(output)

  def clear_cache(self):
        # Call this at the start of each new conversation
    self.k_cache = None
    self.v_cache = None

In [35]:
import torch
import torch.nn as nn

# ============================================================
# Simplified Attention with KV Cache
# This is NOT the full transformer — it's the core idea
# ============================================================

class SimpleAttentionWithKVCache(nn.Module):
    def __init__(self, dim=64, num_heads=4):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads

        # Projection layers for Q, K, V
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)

        # THE KV CACHE — starts empty
        # We will fill this as we generate tokens
        self.k_cache = None  # will store all past Keys
        self.v_cache = None  # will store all past Values

    def forward(self, x):
        # x shape: (batch, seq_len, dim)
        batch, seq_len, _ = x.shape

        # Compute Q, K, V for current token(s)
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # ============================================================
        # THIS IS THE KEY PART — KV CACHE LOGIC
        # If cache exists, APPEND new K and V to existing cache
        # Instead of recomputing everything from scratch
        # ============================================================

        if self.k_cache is not None:
            # Append new keys and values to cache
            # dim=1 means we are appending along the sequence dimension
            k = torch.cat([self.k_cache, k], dim=1)
            v = torch.cat([self.v_cache, v], dim=1)

        # Update cache with current K and V
        self.k_cache = k
        self.v_cache = v

        # Now compute attention using ALL keys and values
        # (current + all previous from cache)
        scale = self.head_dim ** 0.5
        attn_weights = torch.bmm(q, k.transpose(1, 2)) / scale
        attn_weights = torch.softmax(attn_weights, dim=-1)
        output = torch.bmm(attn_weights, v)

        return self.out_proj(output)

    def clear_cache(self):
        # Call this at the start of each new conversation
        self.k_cache = None
        self.v_cache = None



In [36]:

# ============================================================
# Demonstrate KV Cache working
# ============================================================

model = SimpleAttentionWithKVCache(dim=64, num_heads=4)
model.eval()

print("=== Simulating token-by-token generation ===\n")

with torch.no_grad():
    # Token 1 — no cache yet
    token1 = torch.randn(1, 1, 64)
    out1 = model(token1)
    print(f"After token 1 — Cache size: {model.k_cache.shape}")
    # Cache shape: (1, 1, 64) — 1 token stored

    # Token 2 — cache has token 1
    token2 = torch.randn(1, 1, 64)
    out2 = model(token2)
    print(f"After token 2 — Cache size: {model.k_cache.shape}")
    # Cache shape: (1, 2, 64) — 2 tokens stored

    # Token 3 — cache has tokens 1 and 2
    token3 = torch.randn(1, 1, 64)
    out3 = model(token3)
    print(f"After token 3 — Cache size: {model.k_cache.shape}")
    # Cache shape: (1, 3, 64) — 3 tokens stored

print("\nKV cache is growing as we generate more tokens!")
print("Without cache: we would recompute ALL previous tokens every step")
print("With cache: we only compute the NEW token and reuse the rest")

=== Simulating token-by-token generation ===

After token 1 — Cache size: torch.Size([1, 1, 64])
After token 2 — Cache size: torch.Size([1, 2, 64])
After token 3 — Cache size: torch.Size([1, 3, 64])

KV cache is growing as we generate more tokens!
Without cache: we would recompute ALL previous tokens every step
With cache: we only compute the NEW token and reuse the rest
